# Fine-tune Whisper on the corrected Kenyan recordings

GPU on (T4 or P100), Internet on. Attach a **private** dataset that holds, for every recording, the audio, its `... corrections log.json` and `... language labels.csv` from the Transcript Corrector, plus a `recordings.json` that lists them (see `docs/notes/training_plan.md`).

Give every recording a split (`train`, `dev` or `test`) and never split one recording across them.

In [ ]:
!rm -rf transcribe && git clone -q --depth 1 -b matt/relaxed-meitner-2lfrx6 https://github.com/mattobryan/transcribe.git
!cd transcribe && pip install -q -e ".[train]"

### 1. Build the training set (only chunks a person checked)

In [ ]:
DATA = '/kaggle/input/YOUR-DATASET-NAME'   # the folder that holds recordings.json
!cd transcribe && python -m transcribe.train.prepare $DATA/recordings.json --out /kaggle/working/train

### 2. Baselines on the test recording: Scribe, and Whisper before any training

In [ ]:
!cd transcribe && python -m transcribe.train.evaluate /kaggle/working/train/test.jsonl --systems scribe_raw scribe_cleaned hf:openai/whisper-small --out /kaggle/working/reports/baseline.json

### 3. Fine-tune (LoRA). `whisper-small` first; try `openai/whisper-large-v3-turbo` once this works

In [ ]:
!cd transcribe && python -m transcribe.train.finetune /kaggle/working/train --model openai/whisper-small --out /kaggle/working/run1 --epochs 3 --batch 8

### 4. Score the fine-tuned model on the same test recording and pack the results

In [ ]:
!cd transcribe && python -m transcribe.train.evaluate /kaggle/working/train/test.jsonl --systems scribe_cleaned hf:openai/whisper-small hf:openai/whisper-small@/kaggle/working/run1/adapter --out /kaggle/working/reports/after.json
!cd /kaggle/working && zip -qr results.zip run1 reports && echo done